# Task 1 — Direct (no-CoT) answering baseline

CS60075 Assignment 2, Project 8 · BIG-Bench-Hard, logical deduction (five objects) · 150 items, seed 49

**What it does.** Asks for the final answer with no reasoning, greedily, in two variants: zero-shot, and
4-shot with four fixed exemplars whose answers contain only the option letter. The reply is prefilled with
"The answer is" and capped at 16 new tokens, so a direct answer cannot turn into reasoning.

**Reports.** Accuracy with a 95% bootstrap interval for each variant and model, the number of outputs the
parser could not read, and the number that hit the 16-token cap.

**How to run.** With `RUN_MODEL = False` (default) the notebook re-scores the generations committed in
`results/models/<model>/task1/bbh/` and shows the results of both models; no GPU is needed. With
`RUN_MODEL = True` it runs `MODEL` first (a GPU is needed; on Colab add an `HF_TOKEN` secret for Llama).

Shared code (prompts, model loading, answer parsing, statistics) is in `code/common/`.

In [ ]:
# Setup: use this repo if the notebook is opened inside it; otherwise (e.g. on Colab) clone it.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/Chimankarparag/cs60075-cot-faithfulness.git"
here = Path.cwd()
REPO = next((p for p in [here, *here.parents] if (p / "code" / "common" / "config.py").exists()), None)
if REPO is None:
    token = os.environ.get("GITHUB_TOKEN")  # only needed while the repository is private
    url = f"https://{token + '@' if token else ''}{REPO_URL}"
    subprocess.run(["git", "clone", "--depth", "1", url, "cs60075-cot-faithfulness"], check=True)
    REPO = here / "cs60075-cot-faithfulness"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "code"))
print("repo:", REPO)

In [ ]:
MODEL = "llama3.1-8b"   # model to run when RUN_MODEL is True: "llama3.1-8b" or "qwen3-8b"
RUN_MODEL = False       # True: generate with the model (GPU); False: use the committed generations
N_LIMIT = None          # e.g. 10 for a quick test (written to results/smoke/)
BACKEND = "auto"        # "vllm" (KGP servers), "hf" (transformers, e.g. Colab) or "auto"
TP = 1                  # vLLM tensor parallel size

In [ ]:
from dataclasses import asdict

from common.config import DIRECT_DECODING, N_FEWSHOT, SEED, model_spec
from common.data import load_eval, load_fewshot
from common.io_utils import read_json, read_jsonl, results_dir, write_json, write_jsonl
from common.metrics import fmt_ci, summarize_binary
from common.model import Request, load_generator
from common.notebook import model_settings, models_with_results, show_markdown
from common.parsing import is_correct, parse_answer
from common.prompts import direct_prompt
from common.runinfo import RunRecord

SMOKE = bool(N_LIMIT)
VARIANTS = {"zero_shot": 0, "four_shot": N_FEWSHOT}   # variant -> number of exemplars

## Prompting and scoring

In [ ]:
def build_requests(items: list[dict], exemplars: list[dict]) -> list[Request]:
    """One greedy request per item; `exemplars` is empty for zero-shot."""
    return [Request(direct_prompt(item, exemplars), DIRECT_DECODING, SEED) for item in items]


def score_row(item: dict, output_text: str, n_tokens: int, hit_max_tokens: bool) -> dict:
    """Parse one output (first option letter) and compare it with the gold answer."""
    pred = parse_answer(output_text, item["options"])
    return {"id": item["id"], "gold": item["gold"], "output": output_text, "pred": pred,
            "correct": is_correct(pred, item["gold"]), "n_tokens": n_tokens,
            "hit_max_tokens": hit_max_tokens}


def summarize(rows: list[dict]) -> dict:
    """Accuracy with a bootstrap CI, plus parse failures and outputs that hit the 16-token cap
    (a sign the model started explaining instead of answering)."""
    summary = summarize_binary(r["correct"] for r in rows)
    summary["parse_failures"] = sum(r["pred"] is None for r in rows)
    summary["hit_max_tokens"] = sum(r["hit_max_tokens"] for r in rows)
    return summary


def score_task1(model: str, items: list[dict]) -> dict:
    """Re-parse the saved outputs of `model` and write metrics.json and summary.md."""
    out = results_dir(model, "task1", SMOKE)
    by_id = {item["id"]: item for item in items}
    variants = {}
    for variant in VARIANTS:
        rows = [score_row(by_id[r["id"]], r["output"], r["n_tokens"], r["hit_max_tokens"])
                for r in read_jsonl(out / f"{variant}.jsonl")]
        write_jsonl(out / f"{variant}.jsonl", rows)
        variants[variant] = summarize(rows)
    best = max(variants, key=lambda v: variants[v]["mean"])  # a tie keeps zero-shot
    spec = model_spec(model)
    metrics = {"task": "task1", "dataset": "bbh", "n_items": len(items), "model_key": model, "model": spec.hf_id,
               "model_revision": spec.revision, "seed": SEED, "decoding": asdict(DIRECT_DECODING),
               "variants": variants, "best_variant": best}
    write_json(out / "metrics.json", metrics)
    lines = [f"# Task 1 - direct answering (n = {len(items)}, {spec.hf_id})", "",
             "| Variant | Accuracy % [95% CI] | Parse failures | Hit 16-token cap |", "| --- | --- | --- | --- |"]
    for name, s in variants.items():
        lines.append(f"| {name}{' (best)' if name == best else ''} | {fmt_ci(s)} | "
                     f"{s['parse_failures']} | {s['hit_max_tokens']} |")
    (out / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return metrics

## Run the model (only when `RUN_MODEL = True`)

In [ ]:
items = load_eval(N_LIMIT)
if RUN_MODEL:
    settings = model_settings(BACKEND)
    out = results_dir(MODEL, "task1", SMOKE)
    with RunRecord(out, "task1", MODEL, settings["backend"], TP) as record:
        gen = load_generator(MODEL, tp=TP, **settings)
        exemplars = load_fewshot()
        for variant, n_shots in VARIANTS.items():
            with record.stage(variant):
                outputs = gen.generate(build_requests(items, exemplars[:n_shots]), desc=f"task1 {variant}")
            write_jsonl(out / f"{variant}.jsonl",
                        [score_row(item, o.text, o.n_tokens, o.finish_reason == "length")
                         for item, o in zip(items, outputs)])
    del gen
print(f"{len(items)} evaluation items")

## Results

In [ ]:
import pandas as pd

metrics = {m: score_task1(m, items) for m in models_with_results("task1", SMOKE)}
table = pd.DataFrame({
    model_spec(m).hf_id: {f"{v} accuracy % [95% CI]": fmt_ci(s) for v, s in mm["variants"].items()}
                         | {f"{v} parse failures / hit cap": f"{s['parse_failures']} / {s['hit_max_tokens']}"
                            for v, s in mm["variants"].items()}
    for m, mm in metrics.items()})
table

In [ ]:
import matplotlib.pyplot as plt
from common.plots import COLORS, style

fig, ax = plt.subplots(figsize=(5.5, 3))
width = 0.38
for k, (m, mm) in enumerate(metrics.items()):
    xs = [i + (k - 0.5) * width for i in range(len(VARIANTS))]
    means = [100 * mm["variants"][v]["mean"] for v in VARIANTS]
    err = [[100 * (mm["variants"][v]["mean"] - mm["variants"][v]["ci95"][0]) for v in VARIANTS],
           [100 * (mm["variants"][v]["ci95"][1] - mm["variants"][v]["mean"]) for v in VARIANTS]]
    ax.bar(xs, means, width * 0.92, yerr=err, capsize=3, error_kw={'lw': 0.8, 'ecolor': '#52514e'}, color=COLORS[m], label=model_spec(m).hf_id.split("/")[1])
    for x, y in zip(xs, means):
        ax.text(x, y + 9, f"{y:.1f}", ha="center", fontsize=8)
ax.axhline(20, color="#52514e", lw=0.8, ls=":")
ax.set_xticks(range(len(VARIANTS)), ["zero-shot", "4-shot"])
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title("Task 1: direct answering (95% CI; dotted: chance)", loc="left", fontsize=10)
ax.legend(frameon=False, fontsize=8)
style(ax)
plt.show()

**Reading the numbers.** The better variant of each model is the reference point for the CoT gain in Task 2.
Both models are well above chance (20%) without reasoning; a model that can often answer directly has less
reason to rely on its chain of thought, which is what Task 2 measures.